# Analyse Matches

Shows a finished match-analysis run: its KPI tables, charts and example rows, for one source/target/country scenario.

This notebook builds nothing. A run is made by `scripts/analyze_matches.py`, which writes it to `artifacts/analysis/match_analysis/runs/<run_date>/<scenario>/`; this notebook finds the runs on disk, picks one, and reads it back. `analysis.match_metrics` names every file, so nothing here spells a path.

If the scenario has no run yet, make one with:

```bash
uv run --no-sync python scripts/analyze_matches.py --source gleif --target gb
```

To compare many scenarios at once, pass that script `--scenarios-file scenarios.json`.


In [ ]:
import sys
from pathlib import Path

import polars as pl
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import _bootstrap  # noqa: F401

from analysis.match_metrics import (
    resolve_match_analysis_paths,
    resolve_match_analysis_run_root,
)
from workspace.artifact_layout import analysis_report_runs_root
from workspace.roots import default_workspace_roots

ROOTS = default_workspace_roots(ROOT)

# Local notebook configuration (edit directly in this cell).
SOURCE_SYSTEM = "gleif"  # e.g. gleif, wikidata
TARGET_SYSTEM = "gb"  # target system code, e.g. ie, fr
COUNTRY = None  # optional; defaults to TARGET_SYSTEM when None
RUN_DATE = None  # YYYY-MM-DD, or None for the scenario's latest run on disk

print(f"ROOT={ROOT}")
print(f"SOURCE_SYSTEM={SOURCE_SYSTEM} TARGET_SYSTEM={TARGET_SYSTEM} COUNTRY={COUNTRY}")

## 1) Find the run

Lists the dates this scenario has a finished run for, and resolves the chosen run's files.


In [ ]:
country = COUNTRY or TARGET_SYSTEM
# The report's runs sit one directory per run date; the name is match_metrics' own.
runs_root = analysis_report_runs_root(ROOTS, "match_analysis")


def scenario_run_root(run_date: str) -> Path:
    return resolve_match_analysis_run_root(
        ROOTS,
        run_date=run_date,
        source_system=SOURCE_SYSTEM,
        target_system=TARGET_SYSTEM,
        country=country,
    )


run_dates = (
    sorted(
        path.name
        for path in runs_root.iterdir()
        if path.is_dir() and scenario_run_root(path.name).is_dir()
    )
    if runs_root.is_dir()
    else []
)
if not run_dates:
    raise FileNotFoundError(
        f"No match-analysis run on disk for {SOURCE_SYSTEM} -> {TARGET_SYSTEM} "
        f"({country}) under {runs_root}. Make one with:\n"
        f"  uv run --no-sync python scripts/analyze_matches.py "
        f"--source {SOURCE_SYSTEM} --target {TARGET_SYSTEM} --country {country}"
    )
print("Run dates on disk:", ", ".join(run_dates))

run_date = RUN_DATE or run_dates[-1]
if run_date not in run_dates:
    raise ValueError(f"RUN_DATE={run_date!r} names none of the run dates listed above")

# The run's directories exist, so resolving its paths creates nothing.
paths = resolve_match_analysis_paths(
    ROOTS,
    run_date=run_date,
    source_system=SOURCE_SYSTEM,
    target_system=TARGET_SYSTEM,
    country=country,
)
print(f"Scenario: {paths.scenario}")
print(f"Run root: {paths.run_root}")

## 2) Preview KPI Tables

In [ ]:
summary_df = pl.read_parquet(paths.summary_path)
reasons_df = pl.read_parquet(paths.reasons_path)
strategy_df = pl.read_parquet(paths.no_match_strategy_path)

print("Match summary:")
display(summary_df)
print("No-match reasons:")
display(reasons_df)
print("No-match recoverability by strategy:")
display(strategy_df)

## 3) Preview Charts

In [ ]:
chart_paths = [
    paths.match_outcome_chart_path,
    paths.equality_rates_chart_path,
    paths.equality_overlap_venn_path,
    paths.matched_overlap_states_chart_path,
    paths.no_match_reasons_chart_path,
    paths.no_match_recoverability_chart_path,
]

for chart_path in chart_paths:
    print(chart_path.name)
    display(Image(filename=str(chart_path)))

## 4) Inspect Examples

Row-level example tables for deeper triage.

In [ ]:
with pl.Config(tbl_cols=-1, tbl_rows=10, fmt_str_lengths=10000, tbl_width_chars=10000):
    print("Unmatched examples (by reason):")
    display(pl.read_parquet(paths.examples_path).head(20))
    print("Matched name-disagree examples:")
    display(pl.read_parquet(paths.name_disagree_examples_path).head(20))
    print("No-match recoverable by cleansed name (examples):")
    display(pl.read_parquet(paths.no_match_cleansed_examples_path).head(20))

## Interpretation Checklist

- Compare overall match rate and segment rates where name equals `name_cleansed`.
- Inspect top no-match reason codes for the largest actionable causes.
- Review unmatched examples for data quality or matching-rule improvements.
- After a rule change, make a new run with `scripts/analyze_matches.py` and compare its date against the earlier one here.
